# Audio Augmentation Testing

This notebook tests the audio augmentation methods from `data_augmentation.py` on the test audio files.

In [1]:
import os
import sys
import librosa
import numpy as np
import soundfile as sf
from pathlib import Path
import auxiliary as a
import utils as u

# Add the current directory to the path so we can import data_augmentation
sys.path.append('.')

# Import our augmentation functions
from data_augmentation import randomly_eq, apply_compression, add_noise

print("Libraries imported successfully!")

c:\UPF\MIR\MirInharmonicAugmentation\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
c:\UPF\MIR\MirInharmonicAugmentation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully!


In [2]:
# Create output folder
output_folder = 'augmentations_test_outputs'
os.makedirs(output_folder, exist_ok=True)

# List test audio files
test_audio_folder = 'test_audio'
audio_files = [f for f in os.listdir(test_audio_folder) if f.endswith('.mp3')]
print(f"Found {len(audio_files)} audio files in {test_audio_folder}:")
for audio_file in audio_files[:5]:  # Show first 5
    print(f"  - {audio_file}")
if len(audio_files) > 5:
    print(f"  ... and {len(audio_files) - 5} more")

Found 11 audio files in test_audio:
  - TRBFMXE149E3E18222.mp3
  - TRBKNSX149E2E1F89F.mp3
  - TRFVOOM127FA301CFB.mp3
  - TRKINMI127FA68F716.mp3
  - TRMLNPR149E35EF902.mp3
  ... and 6 more


In [ ]:
# Define the augmentations to test
augmentations = [
    ('randomly_eq', randomly_eq, {'gain_range': (-3, 3)}),  # Mild EQ changes
    ('apply_compression', apply_compression, {'threshold_db': -15, 'ratio': 3.0}),  # Moderate compression
    ('add_noise', add_noise, {'noise_type': 'pink', 'snr_db': -25})  # Pink noise at -25dB SNR
]

print("Augmentations to test:")
for name, func, args in augmentations:
    print(f"  - {name}: {args}")

In [ ]:
def print_audio_info(file_path, label="Audio"):
    """Print information about an audio file."""
    try:
        y, sr = librosa.load(file_path, sr=None)
        duration = len(y) / sr
        file_size = os.path.getsize(file_path)
        print(f"{label}: {file_path}")
        print(f"  Duration: {duration:.2f}s, Sample Rate: {sr}Hz, Size: {file_size} bytes")
        return y, sr
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None, None

# Test on first audio file
if audio_files:
    test_file = os.path.join(test_audio_folder, audio_files[0])
    print(f"\n=== Testing on {audio_files[0]} ===")

    # Print original file info
    y_orig, sr_orig = print_audio_info(test_file, "Original")

    # Apply each augmentation
    for aug_name, aug_func, aug_args in augmentations:
        output_file = os.path.join(output_folder, f"{Path(audio_files[0]).stem}_{aug_name}.wav")
        print(f"\nApplying {aug_name}...")

        try:
            aug_func(test_file, output_file, aug_args)
            print_audio_info(output_file, f"Augmented ({aug_name})")
        except Exception as e:
            print(f"Error applying {aug_name}: {e}")

print("\n=== Testing complete for first file ===")

In [ ]:
# Apply all augmentations to all files
print(f"\n=== Applying augmentations to all {len(audio_files)} files ===")

total_processed = 0
for audio_file in audio_files:
    input_path = os.path.join(test_audio_folder, audio_file)
    base_name = Path(audio_file).stem

    print(f"\nProcessing {audio_file}...")

    for aug_name, aug_func, aug_args in augmentations:
        output_file = os.path.join(output_folder, f"{base_name}_{aug_name}.wav")

        try:
            aug_func(input_path, output_file, aug_args)
            total_processed += 1
            print(f"  ✓ {aug_name} -> {os.path.basename(output_file)}")
        except Exception as e:
            print(f"  ✗ {aug_name} failed: {e}")

print(f"\n=== Processing complete ===")
print(f"Total augmentations applied: {total_processed}")
print(f"Output folder: {output_folder}")
print(f"Files created: {len(os.listdir(output_folder))}")

In [ ]:
# List all output files and their information
print("=== Output Files Summary ===")
output_files = sorted(os.listdir(output_folder))
for output_file in output_files:
    file_path = os.path.join(output_folder, output_file)
    print_audio_info(file_path, output_file)

print(f"\nTotal output files: {len(output_files)}")
print("All augmentations completed successfully!")

## Training on Jams


In [3]:
import torch

def check_blackwell():
    print(f"--- Diagnóstico de Hardware ---")
    if torch.cuda.is_available():
        print(f"✅ ¡Conectado a la GPU!")
        print(f"Tarjeta: {torch.cuda.get_device_name(0)}")
        print(f"Capacidad de Cómputo: {torch.cuda.get_device_capability(0)}")
        
        # Test rápido: Multiplicación de matrices en GPU
        x = torch.randn(1000, 1000).cuda()
        y = torch.randn(1000, 1000).cuda()
        z = torch.matmul(x, y)
        print(f"🚀 Test de computación completado en la GPU.")
    else:
        print("❌ Sigue detectando CPU. Revisa que el entorno virtual esté activo.")

if __name__ == "__main__":
    check_blackwell()

--- Diagnóstico de Hardware ---
✅ ¡Conectado a la GPU!
Tarjeta: NVIDIA GeForce RTX 5060 Laptop GPU
Capacidad de Cómputo: (12, 0)
🚀 Test de computación completado en la GPU.


In [ ]:
# Login to Weights & Biases for experiment tracking
import wandb
wandb.login()
import os
#os.environ['WANDB_API_KEY'] = 'api'

In [4]:
JAMS_FOLDER = "jams" # Ensure you set this to the folder where your JAMS files are located
CORPORA = ["MARL-Chords", "Billboard-Chords", "Isophonics"]

df = u.load_chord_data(JAMS_FOLDER)
for each in CORPORA:
	a.print_chord_stats(u.get_chord_stats(df,[each]), title=f"{each} set statistics")

MARL-Chords set statistics
Total chords: 1,322
Unique chords: 165

Billboard-Chords set statistics
Total chords: 5,079
Unique chords: 200

Isophonics set statistics
Total chords: 890
Unique chords: 84



### PreProcess data

### Training cells

In [5]:
from pathlib import Path

DATASET_PREFIX_MAP = {
    "billboard": "Billboard",
    "isophonics": "Isophonics",
    "marl": "MARL",
}

def resolve_existing_dir(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "None of the expected directories exist: " + ", ".join(str(path) for path in candidates)
    )

def normalize_cached_dataset_prefixes(data_dir: Path) -> tuple[int, int]:
    renamed = 0
    skipped = 0

    if not data_dir.exists():
        raise FileNotFoundError(f"Preprocessed data directory not found: {data_dir}")

    for file_path in sorted(data_dir.glob("*.pt")):
        parts = file_path.stem.split("_", 1)
        if len(parts) != 2:
            skipped += 1
            continue

        prefix, remainder = parts
        normalized_prefix = DATASET_PREFIX_MAP.get(prefix.lower())
        if normalized_prefix is None:
            skipped += 1
            continue

        new_name = f"{normalized_prefix}_{remainder}.pt"
        new_path = file_path.with_name(new_name)
        if new_path == file_path:
            continue
        if new_path.exists():
            raise FileExistsError(f"Cannot rename {file_path.name} because {new_path.name} already exists")

        file_path.rename(new_path)
        renamed += 1

    return renamed, skipped

workspace_root = Path.cwd().resolve()
train_root = resolve_existing_dir(workspace_root / "training_jams_data")
augmented_root = resolve_existing_dir(
    workspace_root / "training_plus_augmentation",
    workspace_root / "training_plus_augmentaiton",
)

for label, root in (("base", train_root), ("augmented", augmented_root)):
    renamed, skipped = normalize_cached_dataset_prefixes(root / "preprocessed_data")
    print(f"{label}: normalized {renamed} cached files in {root / 'preprocessed_data'} (skipped {skipped})")

base: normalized 0 cached files in C:\UPF\MIR\MirInharmonicAugmentation\training_jams_data\preprocessed_data (skipped 0)
augmented: normalized 0 cached files in C:\UPF\MIR\MirInharmonicAugmentation\training_plus_augmentaiton\preprocessed_data (skipped 0)


In [6]:
from ace_safe_trainer import main as ace_train_model

# for local
train_data_path = str((train_root / "preprocessed_data").resolve())
vocab_path = str((train_root / "chords_vocab.joblib").resolve())
checkpoint_path = str((train_root / "Checkpoints").resolve())
checkpoint_decomposed_path = str((train_root / "Decomposed_Checkpoints").resolve())

print(f"Base data path: {train_data_path}")

Base data path: C:\UPF\MIR\MirInharmonicAugmentation\training_jams_data\preprocessed_data


In [ ]:
ace_train_model(
    model_name="conformer",
    run_name="colab_conformer_01",
    params={
        "train.data_path": train_data_path,
        "train.vocab_path": vocab_path,
        "train.max_epochs": 10,
        "train.accelerator": "cuda",
        
        "ChocoAudioDataModule.num_workers": 1,
        
        "ModelCheckpoint.dirpath": checkpoint_path,
        "ModelCheckpoint.save_last": True 
    }
)

2026-03-17 15:57:52,974 - root - INFO - system_path_file_exists:ACE/trainer.gin
2026-03-17 15:57:52,974 - root - INFO - gin-config opened resource file:c:\UPF\MIR\MirInharmonicAugmentation\.venv\Lib\site-packages\ACE\trainer.gin
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\User\_netrc.


Found 605 training files and 0 test files.
Training IDs: ['billboard_TRBFMXE149E3E18222', 'billboard_TRBKNSX149E2E1F89F', 'billboard_TRFVOOM127FA301CFB', 'billboard_TRKINMI127FA68F716', 'billboard_TRMLNPR149E35EF902']... 
Test IDs: []... 
Data path: C:\UPF\MIR\MirInharmonicAugmentation\training_jams_data\preprocessed_data
Leakage-safe split: 9 train sources, 2 val sources, 0 test sources


wandb: Currently logged in as: rafaeleduardo-moncayo01 (rafaeleduardo-moncayo01-universitat-pompeu-fabra) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


c:\UPF\MIR\MirInharmonicAugmentation\.venv\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:654: Checkpoint directory C:\UPF\MIR\MirInharmonicAugmentation\training_jams_data\Checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                 | Type               | Params | Mode 
--------------------------------------------------------------------
0 | positional_encodings | PositionalEncoding | 0      | train
1 | input_projection     | Linear             | 37.1 K | train
2 | conformer            | Conformer          | 6.1 M  | train
3 | output_projection    | Linear             | 6.7 K  | train
4 | train_accuracy       | MulticlassAccuracy | 0      | train
5 | val_accuracy         | MulticlassAccuracy | 0      | train
6 | test_accuracy        | MulticlassAccuracy | 0      | train
--------------------------------------------------------------------
6.1 M     Trainable params
0         Non-trainable params
6.1 M     Total params
24.542

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Epoch 9: 100%|██████████| 16/16 [00:00<00:00, 28.66it/s, v_num=eh08, train_accuracy=0.898]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 16/16 [00:00<00:00, 18.24it/s, v_num=eh08, train_accuracy=0.898]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\UPF\MIR\MirInharmonicAugmentation\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.
c:\UPF\MIR\MirInharmonicAugmentation\.venv\Lib\site-packages\lightning\pytorch\utilities\data.py:106: Total length of `DataLoader` across ranks is zero. Please make sure this was your intention.
wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Leakage-safe split: 9 train sources, 2 val sources, 0 test sources


In [ ]:
# Necesary rename the audios after preprocessing from ACE preprocess_data.py 
from pathlib import Path
import shutil

src = augmented_root / "preprocessed_data"
backup = augmented_root / "preprocessed_data_backup_before_marl_to_billboard"
dry_run = False  # pon True para previsualizar sin renombrar

if not src.exists():
    raise FileNotFoundError(f"No existe la carpeta: {src}")

# Backup de seguridad (una sola vez)
if not backup.exists():
    print(f"Creando backup en: {backup}")
    shutil.copytree(src, backup)

renamed = 0
skipped = 0
collisions = 0

for f in sorted(src.glob("*.pt")):
    if not f.stem.startswith("MARL_"):
        skipped += 1
        continue

    new_stem = f"billboard_{f.stem[len('MARL_'):] }"
    new_stem = new_stem.replace(" ", "")  # safety in case of accidental spaces
    new_path = f.with_name(new_stem + ".pt")

    if new_path.exists():
        collisions += 1
        print(f"[COLLISION] {new_path.name} ya existe, se omite {f.name}")
        continue

    if dry_run:
        print(f"[DRY] {f.name}  ->  {new_path.name}")
    else:
        f.rename(new_path)

    renamed += 1

print("\nResumen:")
print(f"  Renombrados MARL_ -> billboard_: {renamed}")
print(f"  Omitidos (no MARL_): {skipped}")
print(f"  Colisiones: {collisions}")
print(f"  Carpeta: {src}")
print(f"  Backup: {backup}")

In [ ]:
train_augmented_data_path = str((augmented_root / "preprocessed_data").resolve())
vocab_path_aug = str((augmented_root / "chords_vocab.joblib").resolve())
checkpoint_path_aug = str((augmented_root / "Checkpoints").resolve())
checkpoint_decomposed_path_aug = str((augmented_root / "Decomposed_Checkpoints").resolve())

print(f"Augmented data path: {train_augmented_data_path}")

In [ ]:
ace_train_model(
    model_name="conformer",
    run_name="colab_conformer_augmented",
    params={
        "train.data_path": train_augmented_data_path,
        "train.vocab_path": vocab_path_aug,
        "train.max_epochs": 10,
        "train.accelerator": "cuda",
        
        "ChocoAudioDataModule.num_workers": 1,
        
        "ModelCheckpoint.dirpath": checkpoint_path_aug,
        "ModelCheckpoint.save_last": True 
    }
)